In [53]:
# Step 1: Setup paths and imports for in2IN and Salsa

import os
import sys
import numpy as np

# In Jupyter, __file__ is not defined; use the current working directory instead.
# This notebook is stored under: .../New_2025/Salsa-Agent/motion_representation/data
NOTEBOOK_DIR = os.getcwd()
print("NOTEBOOK_DIR:", NOTEBOOK_DIR)

# Project root where both `Download/` and `Salsa-Agent/` live is three levels up from `data`
PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, "..", "..", ".."))
print("PROJECT_ROOT:", PROJECT_ROOT)

# Paths to external projects
IN2IN_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, "Download", "in2IN"))
SALSA_ROOT = os.path.abspath(os.path.join(PROJECT_ROOT, "Salsa-Agent"))
print("IN2IN_ROOT:", IN2IN_ROOT)
print("SALSA_ROOT:", SALSA_ROOT)

# Add to sys.path so we can import their modules without modification
if IN2IN_ROOT not in sys.path:
    sys.path.insert(0, IN2IN_ROOT)
if SALSA_ROOT not in sys.path:
    sys.path.insert(0, SALSA_ROOT)

# Now import in2IN utilities (unchanged code)
from in2in.utils.preprocess import load_motion
from in2in.utils.utils import process_motion_interhuman, rigid_transform
from in2in.utils.quaternion import qmul_np, qinv_np, qrot_np

# Basic sanity print so we see that imports worked
print("Loaded in2IN functions:")
print("  load_motion:", load_motion)
print("  process_motion_interhuman:", process_motion_interhuman)
print("  rigid_transform:", rigid_transform)



NOTEBOOK_DIR: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Salsa-Agent/motion_representation/data
PROJECT_ROOT: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025
IN2IN_ROOT: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Download/in2IN
SALSA_ROOT: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Salsa-Agent
Loaded in2IN functions:
  load_motion: <function load_motion at 0x7fbb5818d5a0>
  process_motion_interhuman: <function process_motion_interhuman at 0x7fbb5818d7e0>
  rigid_transform: <function rigid_transform at 0x7fba1e97be20>


In [54]:
# Step 2: Inspect one InterHuman example from the small `motions/1.pkl` file

import os
import pickle

# InterHuman data root (as used by in2IN)
IN2IN_DATA_ROOT = os.path.join(IN2IN_ROOT, "data")
print("IN2IN_DATA_ROOT:", IN2IN_DATA_ROOT)

motions_dir = os.path.join(IN2IN_DATA_ROOT, "motions")
example_pkl = os.path.join(motions_dir, "1.pkl")
print("Loading InterHuman example from:", example_pkl)

with open(example_pkl, "rb") as f:
    interhuman_example = pickle.load(f)

print("Loaded object type:", type(interhuman_example))

if isinstance(interhuman_example, dict):
    print("Dict keys:", list(interhuman_example.keys()))
    for k, v in interhuman_example.items():
        if hasattr(v, "shape"):
            print(f"  key='{k}': shape={v.shape}, dtype={v.dtype}")
        else:
            print(f"  key='{k}': type={type(v)}")


# NOTE: Whatever structure appears here is the *InterHuman representation* that in2IN uses
# after preprocessing. In the next step we will load a Salsa pair and try to build tensors
# with the same shapes/semantics so we can feed them through the same downstream code.

IN2IN_DATA_ROOT: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Download/in2IN/data
Loading InterHuman example from: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Download/in2IN/data/motions/1.pkl
Loaded object type: <class 'dict'>
Dict keys: ['person1', 'person2', 'mocap_framerate', 'frames']
  key='person1': type=<class 'dict'>
  key='person2': type=<class 'dict'>
  key='mocap_framerate': type=<class 'float'>
  key='frames': type=<class 'int'>


In [55]:
# Step 3: Inspect InterHuman `person1` and `person2` motion structure

# This cell assumes `interhuman_example` has already been loaded in Step 2.

assert isinstance(interhuman_example, dict), "Run Step 2 first so `interhuman_example` is available."

person1 = interhuman_example["person1"]
person2 = interhuman_example["person2"]

print("person1 type:", type(person1))
print("person2 type:", type(person2))

if isinstance(person1, dict):
    print("person1 keys:", list(person1.keys()))
    for k, v in person1.items():
        if hasattr(v, "shape"):
            print(f"  person1['{k}']: shape={v.shape}, dtype={v.dtype}")
        else:
            print(f"  person1['{k}']: type={type(v)}")

if isinstance(person2, dict):
    print("person2 keys:", list(person2.keys()))
    for k, v in person2.items():
        if hasattr(v, "shape"):
            print(f"  person2['{k}']: shape={v.shape}, dtype={v.dtype}")
        else:
            print(f"  person2['{k}']: type={type(v)}")


person1 type: <class 'dict'>
person2 type: <class 'dict'>
person1 keys: ['trans', 'root_orient', 'pose_body', 'betas', 'gender']
  person1['trans']: shape=(477, 3), dtype=float32
  person1['root_orient']: shape=(477, 3), dtype=float32
  person1['pose_body']: shape=(477, 63), dtype=float32
  person1['betas']: shape=(10,), dtype=float32
  person1['gender']: type=<class 'str'>
person2 keys: ['trans', 'root_orient', 'pose_body', 'betas', 'gender']
  person2['trans']: shape=(477, 3), dtype=float32
  person2['root_orient']: shape=(477, 3), dtype=float32
  person2['pose_body']: shape=(477, 63), dtype=float32
  person2['betas']: shape=(10,), dtype=float32
  person2['gender']: type=<class 'str'>


In [56]:
# Step 4: Load one Salsa clip and inspect what data we have

import lmdb
import pyarrow

# Path to your Salsa LMDB (clip-level, as you mentioned)
SALSA_LMDB_DIR = os.path.join(PROJECT_ROOT, "Salsa-Agent", "dataset_processed_New", "lmdb_Salsa_pair", "lmdb_train")
print("SALSA_LMDB_DIR:", SALSA_LMDB_DIR)
print("LMDB exists:", os.path.isdir(SALSA_LMDB_DIR))

# Open LMDB and load one video entry (which contains clips)
lmdb_env = lmdb.open(SALSA_LMDB_DIR, readonly=True, lock=False)
with lmdb_env.begin(write=False) as txn:
    # Get first key
    cursor = txn.cursor()
    cursor.first()
    first_key = cursor.key()
    first_value = cursor.value()
    
    # Deserialize
    salsa_video = pyarrow.deserialize(first_value)
    
print("Salsa video type:", type(salsa_video))
if isinstance(salsa_video, dict):
    print("Salsa video keys:", list(salsa_video.keys()))
    for k, v in salsa_video.items():
        if hasattr(v, "shape"):
            print(f"  key='{k}': shape={v.shape}, dtype={v.dtype}")
        elif isinstance(v, list):
            print(f"  key='{k}': list with {len(v)} clips")
        else:
            print(f"  key='{k}': type={type(v)}")

# Extract first clip from the video
if isinstance(salsa_video, dict) and "clips" in salsa_video:
    clips = salsa_video["clips"]
    print(f"\nNumber of clips in this video: {len(clips)}")
    if len(clips) > 0:
        clip = clips[0]
        print("\nFirst clip type:", type(clip))
        if isinstance(clip, dict):
            print("Clip keys:", list(clip.keys()))
            for k, v in clip.items():
                if hasattr(v, "shape"):
                    print(f"  clip['{k}']: shape={v.shape}, dtype={v.dtype}")
                elif isinstance(v, (list, tuple)):
                    print(f"  clip['{k}']: {type(v).__name__} with {len(v)} elements")
                else:
                    print(f"  clip['{k}']: type={type(v)}")
        else:
            print("Clip is not a dict:", clip)
    else:
        print("No clips found in this video")
else:
    print("Video structure doesn't match expected format")

lmdb_env.close()

# NOTE: Based on your `salsa_dataloader.py` and `read_all_salsa_pairs`, each clip should have:
# - 'keypoints3d_L', 'keypoints3d_F': (T, 22, 3) or (T, N_joints, 3) joint positions
# - 'rotmat_L', 'rotmat_F': (T, N_joints, 3, 3) rotation matrices
# - 'HML3D_joints_vec_L', 'HML3D_joints_vec_F': (T, 263) HumanML3D representation
# We need to extract keypoints3d and rotmat to build the InterHuman-style input format.


SALSA_LMDB_DIR: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Salsa-Agent/dataset_processed_New/lmdb_Salsa_pair/lmdb_train
LMDB exists: True
Salsa video type: <class 'dict'>
Salsa video keys: ['vid', 'clips']
  key='vid': type=<class 'str'>
  key='clips': list with 1 clips

Number of clips in this video: 1

First clip type: <class 'dict'>
Clip keys: ['raw_euler_poses_L', 'raw_trans_L', 'keypoints3d_L', 'rotmat_L', 'HML3D_joints_L', 'HML3D_joints_vec_L', 'raw_euler_poses_F', 'raw_trans_F', 'keypoints3d_F', 'rotmat_F', 'HML3D_joints_F', 'HML3D_joints_vec_F', 'audio_raw', 'audio_sr', 'annotation', 'body_betas', 'body_vertices', 'body_faces', 'body_gender']
  clip['raw_euler_poses_L']: shape=(3015, 165), dtype=float64
  clip['raw_trans_L']: shape=(3015, 3), dtype=float64
  clip['keypoints3d_L']: shape=(3015, 22, 3), dtype=float32
  clip['rotmat_L']: shape=(3015, 498), dtype=float64
  clip['HML3D_joints_L']: shape=(3014, 22, 3), dtype=float32
  clip['HM

In [57]:
# Step 5: Convert Salsa clip to InterHuman input format [22*3 positions | 21*6 rotations]

# Based on salsa_smplx_to_rotmat: rotmat is [trans (3) | flattened_rotmats (55*9=495)] = 498 dims
# We need: [22*3 positions | 21*6 rotations] = [66 | 126] = 192 dims

print("Converting Salsa data to InterHuman format...")
print(f"  rotmat structure: [trans (3) | flattened_rotmats (55*9=495)] = 498 dims")

# Extract a small segment for testing
T = min(100, clip['keypoints3d_L'].shape[0])
print(f"\nUsing first {T} frames for conversion test")

keypoints3d_L_sample = clip['keypoints3d_L'][:T].astype(np.float32)  # (T, 22, 3)
keypoints3d_F_sample = clip['keypoints3d_F'][:T].astype(np.float32)  # (T, 22, 3)
rotmat_L_sample = clip['rotmat_L'][:T].astype(np.float32)  # (T, 498)
rotmat_F_sample = clip['rotmat_F'][:T].astype(np.float32)  # (T, 498)

# Step 1: Convert keypoints3d to (T, 22*3) = (T, 66)
positions_L = keypoints3d_L_sample.reshape(T, -1)  # (T, 66)
positions_F = keypoints3d_F_sample.reshape(T, -1)  # (T, 66)
print(f"\nStep 1 - Positions:")
print(f"  positions_L: {positions_L.shape}")

# Step 2: Extract rotations from rotmat and convert to 6D
# rotmat format: [trans (3) | joint0_rotmat (9) | joint1_rotmat (9) | ... | joint54_rotmat (9)]
# We need the first 21 body joints (excluding root) -> joints 1-21

# Helper function to convert 3x3 rotation matrix to 6D representation
def matrix_to_rotation_6d(rotmat_3x3):
    """Convert 3x3 rotation matrix to 6D representation (first 2 columns)."""
    return rotmat_3x3[:, :2].reshape(-1, 6)

# Extract rotation matrices from rotmat (skip first 3 dims which are translation)
rotmats_flat_L = rotmat_L_sample[:, 3:]  # (T, 495) - skip trans
rotmats_flat_F = rotmat_F_sample[:, 3:]  # (T, 495)

# Reshape to (T, 55, 3, 3)
rotmats_L = rotmats_flat_L.reshape(T, 55, 3, 3)
rotmats_F = rotmats_flat_F.reshape(T, 55, 3, 3)
print(f"\nStep 2 - Rotation matrices:")
print(f"  rotmats_L reshaped: {rotmats_L.shape}")

# Extract first 21 body joints (joints 1-21, excluding root joint 0)
# Note: This assumes SMPL joint ordering matches InterHuman's 22-joint structure
body_rotmats_L = rotmats_L[:, 1:22, :, :]  # (T, 21, 3, 3)
body_rotmats_F = rotmats_F[:, 1:22, :, :]  # (T, 21, 3, 3)

# Convert each 3x3 matrix to 6D representation (first 2 columns)
rotations_6d_L = body_rotmats_L[:, :, :, :2].reshape(T, 21, 6)  # (T, 21, 6)
rotations_6d_F = body_rotmats_F[:, :, :, :2].reshape(T, 21, 6)  # (T, 21, 6)

# Flatten to (T, 126)
rotations_6d_L = rotations_6d_L.reshape(T, -1)  # (T, 126)
rotations_6d_F = rotations_6d_F.reshape(T, -1)  # (T, 126)
print(f"  rotations_6d_L: {rotations_6d_L.shape}")

# Step 3: Concatenate positions and rotations to get InterHuman format
motion_L = np.concatenate([positions_L, rotations_6d_L], axis=-1)  # (T, 192)
motion_F = np.concatenate([positions_F, rotations_6d_F], axis=-1)  # (T, 192)
print(f"\nStep 3 - Final InterHuman format:")
print(f"  motion_L: {motion_L.shape} (should be (T, 192))")
print(f"  motion_F: {motion_F.shape} (should be (T, 192))")

# Save for next step
salsa_motion_L = motion_L
salsa_motion_F = motion_F


Converting Salsa data to InterHuman format...
  rotmat structure: [trans (3) | flattened_rotmats (55*9=495)] = 498 dims

Using first 100 frames for conversion test

Step 1 - Positions:
  positions_L: (100, 66)

Step 2 - Rotation matrices:
  rotmats_L reshaped: (100, 55, 3, 3)
  rotations_6d_L: (100, 126)

Step 3 - Final InterHuman format:
  motion_L: (100, 192) (should be (T, 192))
  motion_F: (100, 192) (should be (T, 192))


In [58]:
# Step 6: Process converted Salsa motions through in2IN's pipeline and compare with InterHuman

print("Processing Salsa motions through in2IN's process_motion_interhuman...")

# Run the same canonicalization as in InterHuman.__getitem__
# This converts raw [22*3 positions | 21*6 rotations] into the InterHuman 'global' representation
salsa_motion1_proc, salsa_root_quat_init1, salsa_root_pos_init1 = process_motion_interhuman(
    salsa_motion_L, 0.001, 0, n_joints=22
)
salsa_motion2_proc, salsa_root_quat_init2, salsa_root_pos_init2 = process_motion_interhuman(
    salsa_motion_F, 0.001, 0, n_joints=22
)

print("Processed motion shapes (after process_motion_interhuman):")
print(f"  salsa_motion1_proc: {salsa_motion1_proc.shape}")
print(f"  salsa_motion2_proc: {salsa_motion2_proc.shape}")

# Compute relative rotation+translation and rigidly transform person 2, as in InterHuman.__getitem__
salsa_r_relative = qmul_np(salsa_root_quat_init2, qinv_np(salsa_root_quat_init1))
salsa_angle = np.arctan2(salsa_r_relative[:, 2:3], salsa_r_relative[:, 0:1])  # relative yaw
salsa_xz = qrot_np(salsa_root_quat_init1, salsa_root_pos_init2 - salsa_root_pos_init1)[:, [0, 2]]  # relative XZ
salsa_relative = np.concatenate([salsa_angle, salsa_xz], axis=-1)[0]  # (3,) [yaw, x, z]

salsa_motion2_rel = rigid_transform(salsa_relative, salsa_motion2_proc)

print("\nRelative transform (yaw, x, z):", salsa_relative)
print("Final shapes after rigid_transform:")
print(f"  salsa_motion1_proc: {salsa_motion1_proc.shape}")
print(f"  salsa_motion2_rel: {salsa_motion2_rel.shape}")

# Quick numeric sanity check: compare first-frame root positions in canonical frame
salsa_root1_pos0 = salsa_motion1_proc[0, :22*3].reshape(22, 3)[0]
salsa_root2_pos0 = salsa_motion2_rel[0, :22*3].reshape(22, 3)[0]
print("\nRoot joint positions at first frame (canonical frame):")
print(f"  salsa person1 root: {salsa_root1_pos0}")
print(f"  salsa person2 root (after relative alignment): {salsa_root2_pos0}")

# Compare with InterHuman example (if we had processed one)
print("\n" + "="*60)
print("SUCCESS: Salsa data successfully converted and processed!")
print("="*60)
print("Next steps:")
print("  1. Visualize both InterHuman and Salsa pairs using in2IN's visualization")
print("  2. Verify that canonicalization (floor, origin, facing +Z) is consistent")
print("  3. Check that relative positioning between person1 and person2 makes sense")


Processing Salsa motions through in2IN's process_motion_interhuman...
Processed motion shapes (after process_motion_interhuman):
  salsa_motion1_proc: (99, 262)
  salsa_motion2_proc: (99, 262)

Relative transform (yaw, x, z): [1.5295483  0.1919772  0.01631661]
Final shapes after rigid_transform:
  salsa_motion1_proc: (99, 262)
  salsa_motion2_rel: (99, 262)

Root joint positions at first frame (canonical frame):
  salsa person1 root: [0.        0.5182469 0.       ]
  salsa person2 root (after relative alignment): [0.1919772  0.30676508 0.01631661]

SUCCESS: Salsa data successfully converted and processed!
Next steps:
  1. Visualize both InterHuman and Salsa pairs using in2IN's visualization
  2. Verify that canonicalization (floor, origin, facing +Z) is consistent
  3. Check that relative positioning between person1 and person2 makes sense


In [59]:
# Step 7a: Convert InterHuman SMPL params to InterHuman format (replicating preprocessing pipeline)
#
# This step attempts to replicate in2IN's preprocessing from raw SMPL parameters.
# NOTE: The preprocessed .npy files were created using SMPL's actual forward kinematics,
# which we cannot easily replicate without the SMPL model. However, we can use in2IN's
# skeleton to get an approximation and compare with the preprocessed files.
#
# This is useful for:
# 1. Verifying our understanding of the preprocessing pipeline
# 2. Comparing with preprocessed files to identify any differences
# 3. Understanding how to convert Salsa SMPL data to InterHuman format

from in2in.utils.skeleton import Skeleton
from in2in.utils.paramUtil import HML_KINEMATIC_CHAIN, HML_RAW_OFFSETS
from in2in.utils.quaternion import expmap_to_quaternion, quaternion_to_cont6d_np
import torch

print("="*60)
print("Step 7a: Converting InterHuman SMPL params to InterHuman format")
print("="*60)
print("NOTE: This uses in2IN's skeleton (approximation of SMPL's FK)")
print("      The preprocessed .npy files use SMPL's actual FK (ground truth)")

# InterHuman example has SMPL params: trans, root_orient (axis-angle), pose_body (21*3 axis-angle)
person1 = interhuman_example["person1"]
person2 = interhuman_example["person2"]

T_ih_smpl = person1['trans'].shape[0]
T_salsa = salsa_motion1_proc.shape[0]  # From Step 6
print(f"InterHuman SMPL motion length: {T_ih_smpl} frames")
print(f"Salsa motion length: {T_salsa} frames")

# Use a subset for comparison
T_compare_smpl = min(T_ih_smpl, T_salsa, 100)
print(f"Using {T_compare_smpl} frames for comparison")

# Extract SMPL parameters
trans1_smpl = person1['trans'][:T_compare_smpl].astype(np.float32)  # (T, 3)
root_orient1_smpl = person1['root_orient'][:T_compare_smpl].astype(np.float32)  # (T, 3) axis-angle
pose_body1_smpl = person1['pose_body'][:T_compare_smpl].astype(np.float32)  # (T, 63) = 21*3 axis-angle

trans2_smpl = person2['trans'][:T_compare_smpl].astype(np.float32)
root_orient2_smpl = person2['root_orient'][:T_compare_smpl].astype(np.float32)
pose_body2_smpl = person2['pose_body'][:T_compare_smpl].astype(np.float32)

# Convert axis-angle (exponential map) to quaternions using in2IN's function
root_quat1_smpl = expmap_to_quaternion(root_orient1_smpl)  # (T, 4)
root_quat2_smpl = expmap_to_quaternion(root_orient2_smpl)  # (T, 4)

# Convert body pose (21 joints * 3 axis-angle) to quaternions
pose_body1_reshaped_smpl = pose_body1_smpl.reshape(T_compare_smpl, 21, 3)  # (T, 21, 3)
pose_body2_reshaped_smpl = pose_body2_smpl.reshape(T_compare_smpl, 21, 3)  # (T, 21, 3)
body_quat1_smpl = expmap_to_quaternion(pose_body1_reshaped_smpl)  # (T, 21, 4)
body_quat2_smpl = expmap_to_quaternion(pose_body2_reshaped_smpl)  # (T, 21, 4)

# Combine root and body quaternions: (T, 22, 4) where first is root
quat_params1_smpl = np.concatenate([root_quat1_smpl[:, None, :], body_quat1_smpl], axis=1)  # (T, 22, 4)
quat_params2_smpl = np.concatenate([root_quat2_smpl[:, None, :], body_quat2_smpl], axis=1)  # (T, 22, 4)

# Use in2IN's Skeleton forward kinematics to get joint positions
# NOTE: This is an approximation - preprocessed files use SMPL's actual FK
n_raw_offsets = torch.from_numpy(HML_RAW_OFFSETS)
skeleton = Skeleton(n_raw_offsets, HML_KINEMATIC_CHAIN, "cpu")
skeleton._offset = skeleton._raw_offset.clone()
joints1_smpl = skeleton.forward_kinematics_np(quat_params1_smpl, trans1_smpl)  # (T, 22, 3)
joints2_smpl = skeleton.forward_kinematics_np(quat_params2_smpl, trans2_smpl)  # (T, 22, 3)

print(f"  InterHuman joints1 (from SMPL) shape: {joints1_smpl.shape}")
print(f"  InterHuman joints2 (from SMPL) shape: {joints2_smpl.shape}")

# Convert body quaternions to 6D rotations using in2IN's function
body_6d1_smpl = quaternion_to_cont6d_np(body_quat1_smpl)  # (T, 21, 6)
body_6d2_smpl = quaternion_to_cont6d_np(body_quat2_smpl)  # (T, 21, 6)
body_6d1_flat_smpl = body_6d1_smpl.reshape(T_compare_smpl, -1)  # (T, 126)
body_6d2_flat_smpl = body_6d2_smpl.reshape(T_compare_smpl, -1)  # (T, 126)

# Flatten joint positions
positions1_flat_smpl = joints1_smpl.reshape(T_compare_smpl, -1)  # (T, 66)
positions2_flat_smpl = joints2_smpl.reshape(T_compare_smpl, -1)  # (T, 66)

# Build InterHuman format: [22*3 positions | 21*6 rotations]
ih_motion1_smpl = np.concatenate([positions1_flat_smpl, body_6d1_flat_smpl], axis=-1)  # (T, 192)
ih_motion2_smpl = np.concatenate([positions2_flat_smpl, body_6d2_flat_smpl], axis=-1)  # (T, 192)

print(f"  InterHuman motion1 (from SMPL) format: {ih_motion1_smpl.shape} (should be (T, 192))")
print(f"  InterHuman motion2 (from SMPL) format: {ih_motion2_smpl.shape} (should be (T, 192))")
print(f"\n  NOTE: This is an approximation using in2IN's skeleton.")
print(f"        Preprocessed .npy files use SMPL's actual FK (ground truth).")


Step 7a: Converting InterHuman SMPL params to InterHuman format
NOTE: This uses in2IN's skeleton (approximation of SMPL's FK)
      The preprocessed .npy files use SMPL's actual FK (ground truth)
InterHuman SMPL motion length: 477 frames
Salsa motion length: 99 frames
Using 99 frames for comparison
  InterHuman joints1 (from SMPL) shape: (99, 22, 3)
  InterHuman joints2 (from SMPL) shape: (99, 22, 3)
  InterHuman motion1 (from SMPL) format: (99, 192) (should be (T, 192))
  InterHuman motion2 (from SMPL) format: (99, 192) (should be (T, 192))

  NOTE: This is an approximation using in2IN's skeleton.
        Preprocessed .npy files use SMPL's actual FK (ground truth).


In [60]:
# Step 7b: Load InterHuman preprocessed data (ground truth format)
#
# STRATEGY: Instead of using in2IN's skeleton (which produces wrong results), we'll:
# 1. Load a preprocessed .npy file to see what the correct format should be
# 2. Compare it with what we generate to identify the issue
# 3. Use the preprocessed file directly for InterHuman (since we can't easily replicate SMPL's FK)
#
# NOTE: The user has raw SMPL data in their LMDB, so for Salsa we can use SMPL's actual FK.
# For InterHuman, we'll use the preprocessed .npy files that in2IN provides, which were
# created using SMPL's actual forward kinematics.

from in2in.utils.preprocess import load_motion
import os

print("="*60)
print("Step 7b: Loading InterHuman preprocessed data (ground truth)")
print("="*60)
print("Using in2IN's load_motion to get the correct InterHuman format")
print("(Preprocessed files were created using SMPL's actual forward kinematics)")

# Load preprocessed InterHuman data directly (this is the correct format created by in2IN)
# The preprocessed .npy files were created using SMPL's actual forward kinematics
IN2IN_DATA_ROOT = os.path.join(IN2IN_ROOT, "data")
motions_processed_dir_p1 = os.path.join(IN2IN_DATA_ROOT, "motions_processed", "person1")
motions_processed_dir_p2 = os.path.join(IN2IN_DATA_ROOT, "motions_processed", "person2")

if not os.path.exists(motions_processed_dir_p1):
    raise FileNotFoundError(f"Preprocessed motions directory not found: {motions_processed_dir_p1}")

# Find a preprocessed file that matches our example (try to find "1.npy" corresponding to "1.pkl")
npy_files = [f for f in os.listdir(motions_processed_dir_p1) if f.endswith('.npy')]
if len(npy_files) == 0:
    raise FileNotFoundError(f"No .npy files found in {motions_processed_dir_p1}")

# Try to find "1.npy" first (to match "1.pkl"), otherwise use the first available file
example_file = "1.npy" if "1.npy" in npy_files else npy_files[0]
ih_file_p1 = os.path.join(motions_processed_dir_p1, example_file)
ih_file_p2 = os.path.join(motions_processed_dir_p2, example_file)

print(f"Loading preprocessed InterHuman motions:")
print(f"  Person1: {ih_file_p1}")
print(f"  Person2: {ih_file_p2}")

# Use in2IN's load_motion function (exactly as they do in their dataset)
ih_motion1_full, _ = load_motion(ih_file_p1, min_length=1, swap=False)
ih_motion2_full, _ = load_motion(ih_file_p2, min_length=1, swap=False)

if ih_motion1_full is None or ih_motion2_full is None:
    raise ValueError("Failed to load InterHuman preprocessed motions")

print(f"  Loaded InterHuman motion1 shape: {ih_motion1_full.shape}")
print(f"  Loaded InterHuman motion2 shape: {ih_motion2_full.shape}")

# These are already in [22*3 positions | 21*6 rotations] format (192 dims)
T_ih = ih_motion1_full.shape[0]
T_salsa = salsa_motion1_proc.shape[0]  # From Step 6
print(f"\nInterHuman motion length: {T_ih} frames")
print(f"Salsa motion length: {T_salsa} frames")

# Use a subset for comparison (same length as Salsa)
T_compare = min(T_ih, T_salsa, 100)  # Use 100 frames for visualization
print(f"Using {T_compare} frames for comparison")

# Extract the same length from both
ih_motion1_npy = ih_motion1_full[:T_compare]
ih_motion2_npy = ih_motion2_full[:T_compare]

print(f"  InterHuman motion1 (from .npy) format: {ih_motion1_npy.shape} (should be (T, 192))")
print(f"  InterHuman motion2 (from .npy) format: {ih_motion2_npy.shape} (should be (T, 192))")
print(f"\n  NOTE: Using preprocessed .npy files (created with SMPL's actual FK)")
print(f"        This is the ground truth format used by in2IN for training.")

# Now process both InterHuman (from .npy) and Salsa through the SAME pipeline
print("\n" + "="*60)
print("Processing InterHuman (from .npy) and Salsa through process_motion_interhuman...")
print("="*60)

# Process InterHuman motions (from preprocessed .npy files)
ih_motion1_proc_npy, ih_root_quat_init1_npy, ih_root_pos_init1_npy = process_motion_interhuman(
    ih_motion1_npy, 0.001, 0, n_joints=22
)
ih_motion2_proc_npy, ih_root_quat_init2_npy, ih_root_pos_init2_npy = process_motion_interhuman(
    ih_motion2_npy, 0.001, 0, n_joints=22
)

# Compute relative transform for InterHuman (from .npy)
ih_r_relative_npy = qmul_np(ih_root_quat_init2_npy, qinv_np(ih_root_quat_init1_npy))
ih_angle_npy = np.arctan2(ih_r_relative_npy[:, 2:3], ih_r_relative_npy[:, 0:1])
ih_xz_npy = qrot_np(ih_root_quat_init1_npy, ih_root_pos_init2_npy - ih_root_pos_init1_npy)[:, [0, 2]]
ih_relative_npy = np.concatenate([ih_angle_npy, ih_xz_npy], axis=-1)[0]
ih_motion2_rel_npy = rigid_transform(ih_relative_npy, ih_motion2_proc_npy)

# Also process InterHuman from SMPL (for comparison)
ih_motion1_proc_smpl, ih_root_quat_init1_smpl, ih_root_pos_init1_smpl = process_motion_interhuman(
    ih_motion1_smpl, 0.001, 0, n_joints=22
)
ih_motion2_proc_smpl, ih_root_quat_init2_smpl, ih_root_pos_init2_smpl = process_motion_interhuman(
    ih_motion2_smpl, 0.001, 0, n_joints=22
)

# Compute relative transform for InterHuman (from SMPL)
ih_r_relative_smpl = qmul_np(ih_root_quat_init2_smpl, qinv_np(ih_root_quat_init1_smpl))
ih_angle_smpl = np.arctan2(ih_r_relative_smpl[:, 2:3], ih_r_relative_smpl[:, 0:1])
ih_xz_smpl = qrot_np(ih_root_quat_init1_smpl, ih_root_pos_init2_smpl - ih_root_pos_init1_smpl)[:, [0, 2]]
ih_relative_smpl = np.concatenate([ih_angle_smpl, ih_xz_smpl], axis=-1)[0]
ih_motion2_rel_smpl = rigid_transform(ih_relative_smpl, ih_motion2_proc_smpl)

# Salsa motions: extract the same length BEFORE processing (since process_motion_interhuman reduces by 1)
salsa_motion1_raw_compare = salsa_motion_L[:T_compare]
salsa_motion2_raw_compare = salsa_motion_F[:T_compare]

# Process Salsa motions through the same pipeline as InterHuman (using in2IN's process_motion_interhuman)
salsa_motion1_proc_compare, salsa_root_quat_init1_comp, salsa_root_pos_init1_comp = process_motion_interhuman(
    salsa_motion1_raw_compare, 0.001, 0, n_joints=22
)
salsa_motion2_proc_compare, salsa_root_quat_init2_comp, salsa_root_pos_init2_comp = process_motion_interhuman(
    salsa_motion2_raw_compare, 0.001, 0, n_joints=22
)

# Compute relative transform for Salsa (same as InterHuman, using in2IN's functions)
salsa_r_relative_comp = qmul_np(salsa_root_quat_init2_comp, qinv_np(salsa_root_quat_init1_comp))
salsa_angle_comp = np.arctan2(salsa_r_relative_comp[:, 2:3], salsa_r_relative_comp[:, 0:1])
salsa_xz_comp = qrot_np(salsa_root_quat_init1_comp, salsa_root_pos_init2_comp - salsa_root_pos_init1_comp)[:, [0, 2]]
salsa_relative_comp = np.concatenate([salsa_angle_comp, salsa_xz_comp], axis=-1)[0]
salsa_motion2_rel_compare = rigid_transform(salsa_relative_comp, salsa_motion2_proc_compare)

print(f"  InterHuman (from .npy) processed: {ih_motion1_proc_npy.shape}, {ih_motion2_rel_npy.shape}")
print(f"  InterHuman (from SMPL) processed: {ih_motion1_proc_smpl.shape}, {ih_motion2_rel_smpl.shape}")
print(f"  Salsa processed: {salsa_motion1_proc_compare.shape}, {salsa_motion2_rel_compare.shape}")

# Extract joint positions for visualization
# Note: process_motion_interhuman reduces length by 1, so use actual processed lengths
T_ih_proc_npy = ih_motion1_proc_npy.shape[0]
T_ih_proc_smpl = ih_motion1_proc_smpl.shape[0]
T_salsa_proc = salsa_motion1_proc_compare.shape[0]
# Use the minimum length for comparison
T_viz = min(T_ih_proc_npy, T_ih_proc_smpl, T_salsa_proc)

# Joints from preprocessed .npy files (ground truth)
ih_joints1_npy = ih_motion1_proc_npy[:T_viz, :22*3].reshape(T_viz, 22, 3)
ih_joints2_npy = ih_motion2_rel_npy[:T_viz, :22*3].reshape(T_viz, 22, 3)

# Joints from SMPL conversion (for comparison)
ih_joints1_smpl = ih_motion1_proc_smpl[:T_viz, :22*3].reshape(T_viz, 22, 3)
ih_joints2_smpl = ih_motion2_rel_smpl[:T_viz, :22*3].reshape(T_viz, 22, 3)

# Salsa joints
salsa_joints1 = salsa_motion1_proc_compare[:T_viz, :22*3].reshape(T_viz, 22, 3)
salsa_joints2 = salsa_motion2_rel_compare[:T_viz, :22*3].reshape(T_viz, 22, 3)

print(f"\n  Using {T_viz} frames for visualization (after processing)")

print("\n✓ All datasets processed through the same pipeline!")
print("  - InterHuman from .npy (ground truth)")
print("  - InterHuman from SMPL (approximation)")
print("  - Salsa from keypoints3d/rotmat")


Step 7b: Loading InterHuman preprocessed data (ground truth)
Using in2IN's load_motion to get the correct InterHuman format
(Preprocessed files were created using SMPL's actual forward kinematics)
Loading preprocessed InterHuman motions:
  Person1: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Download/in2IN/data/motions_processed/person1/1.npy
  Person2: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Download/in2IN/data/motions_processed/person2/1.npy
  Loaded InterHuman motion1 shape: (239, 192)
  Loaded InterHuman motion2 shape: (239, 192)

InterHuman motion length: 239 frames
Salsa motion length: 99 frames
Using 99 frames for comparison
  InterHuman motion1 (from .npy) format: (99, 192) (should be (T, 192))
  InterHuman motion2 (from .npy) format: (99, 192) (should be (T, 192))

  NOTE: Using preprocessed .npy files (created with SMPL's actual FK)
        This is the ground truth format used by in2IN for tra

In [ ]:
# NOTE: Conclusion section moved to the end of the notebook (after Step 8 visualization)
# The actual conclusion functions are defined at the very end of the notebook.



CONCLUSION: Salsa to InterHuman conversion functions defined
Functions available:
  1. salsa_smpl_to_interhuman() - Convert single person
  2. salsa_pair_to_interhuman() - Convert pair with relative alignment

These functions use in2IN's functions for consistency.


In [ ]:
# NOTE: Conclusion test moved to the end of the notebook (after Step 8 visualization)

print("="*60)
print("Testing salsa_pair_to_interhuman() with keypoints3d/rotmat")
print("="*60)

# Extract keypoints3d and rotmat from the clip (first 100 frames for testing)
T_test = min(100, clip['keypoints3d_L'].shape[0])
keypoints3d_L_test = clip['keypoints3d_L'][:T_test]  # (T, 22, 3)
rotmat_L_test = clip['rotmat_L'][:T_test]  # (T, 498)
keypoints3d_F_test = clip['keypoints3d_F'][:T_test]  # (T, 22, 3)
rotmat_F_test = clip['rotmat_F'][:T_test]  # (T, 498)

print(f"Input shapes:")
print(f"  keypoints3d_L: {keypoints3d_L_test.shape}")
print(f"  rotmat_L: {rotmat_L_test.shape}")
print(f"  keypoints3d_F: {keypoints3d_F_test.shape}")
print(f"  rotmat_F: {rotmat_F_test.shape}")

# Convert using the conclusion function
salsa_motion1_final, salsa_motion2_final = salsa_pair_to_interhuman(
    keypoints3d_L_test, rotmat_L_test,
    keypoints3d_F_test, rotmat_F_test
)

print(f"\nOutput shapes:")
print(f"  salsa_motion1_final: {salsa_motion1_final.shape}")
print(f"  salsa_motion2_final: {salsa_motion2_final.shape}")

# Extract joints for visualization
salsa_joints1_final = salsa_motion1_final[:, :22*3].reshape(-1, 22, 3)
salsa_joints2_final = salsa_motion2_final[:, :22*3].reshape(-1, 22, 3)

print(f"\nJoint positions for visualization:")
print(f"  salsa_joints1_final: {salsa_joints1_final.shape}")
print(f"  salsa_joints2_final: {salsa_joints2_final.shape}")

# Visualize using in2IN's plot function
from in2in.utils.plot import plot_3d_motion
from in2in.utils.paramUtil import HML_KINEMATIC_CHAIN

viz_path_final = os.path.join(viz_dir, "salsa_pair_conclusion.mp4")
print(f"\nGenerating visualization: {viz_path_final}")
plot_3d_motion(
    save_path=viz_path_final,
    kinematic_tree=HML_KINEMATIC_CHAIN,
    mp_joints=[salsa_joints1_final, salsa_joints2_final],
    title="Salsa Pair (via conclusion function)",
    figsize=(12, 12),
    fps=30,
    radius=6
)

print(f"\n✓ Visualization saved: {viz_path_final}")
print("\n" + "="*60)
print("CONCLUSION COMPLETE")
print("="*60)
print("You can now use salsa_pair_to_interhuman() in your dataloader")
print("to convert Salsa pairs to InterHuman representation consistently.")
print("This function uses keypoints3d/rotmat (the approach that worked) and")
print("in2IN's functions (process_motion_interhuman, rigid_transform) for consistency.")


Testing salsa_pair_to_interhuman() with raw Salsa data
Input shapes:
  raw_euler_poses_L: (100, 165)
  raw_trans_L: (100, 3)
  raw_euler_poses_F: (100, 165)
  raw_trans_F: (100, 3)

Output shapes:
  salsa_motion1_final: (99, 262)
  salsa_motion2_final: (99, 262)

Joint positions for visualization:
  salsa_joints1_final: (99, 22, 3)
  salsa_joints2_final: (99, 22, 3)

Generating visualization: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Salsa-Agent/motion_representation/data/visualizations/salsa_pair_from_raw_smpl.mp4






  0%|          | 0/100 [01:33<?, ?it/s]


























































100%|██████████| 100/100 [00:04<00:00, 24.26it/s]


✓ Visualization saved: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Salsa-Agent/motion_representation/data/visualizations/salsa_pair_from_raw_smpl.mp4

CONCLUSION COMPLETE
You can now use salsa_pair_to_interhuman() in your dataloader
to convert Salsa pairs to InterHuman representation consistently.


# Conclusion: Simple Function to Convert Salsa Raw SMPL to InterHuman Representation

This cell provides a clean, reusable function that converts raw Salsa SMPL data to InterHuman representation,
using in2IN's functions to ensure consistency with their preprocessing pipeline.


In [63]:
# Simple function to convert Salsa raw SMPL data to InterHuman representation
# This function can be used in your dataloader

import numpy as np
import torch
from in2in.utils.skeleton import Skeleton
from in2in.utils.paramUtil import HML_KINEMATIC_CHAIN, HML_RAW_OFFSETS
from in2in.utils.quaternion import expmap_to_quaternion, quaternion_to_cont6d_np
from in2in.utils.utils import process_motion_interhuman, rigid_transform
from in2in.utils.quaternion import qmul_np, qinv_np, qrot_np


def salsa_smpl_to_interhuman(raw_euler_poses, raw_trans, use_smpl_fk=True):
    """
    Convert raw Salsa SMPL data to InterHuman representation.
    
    Args:
        raw_euler_poses: (T, 165) - SMPL pose parameters (axis-angle)
                         First 3 dims: root_orient, next 63 dims: pose_body
        raw_trans: (T, 3) - Global translation
        use_smpl_fk: bool - If True, use SMPL's actual FK (requires SMPL model).
                         If False, use in2IN's skeleton (approximation).
    
    Returns:
        motion_interhuman: (T-1, 262) - Processed InterHuman representation
                          [22*3 positions | 21*6 rotations | velocities | contacts]
    """
    T = raw_euler_poses.shape[0]
    
    # Extract SMPL parameters (same structure as InterHuman)
    root_orient = raw_euler_poses[:, :3].astype(np.float32)  # (T, 3) axis-angle
    pose_body = raw_euler_poses[:, 3:66].astype(np.float32)  # (T, 63) = 21*3 axis-angle
    trans = raw_trans.astype(np.float32)  # (T, 3)
    
    # Convert axis-angle to quaternions (using in2IN's function)
    root_quat = expmap_to_quaternion(root_orient)  # (T, 4)
    pose_body_reshaped = pose_body.reshape(T, 21, 3)  # (T, 21, 3)
    body_quat = expmap_to_quaternion(pose_body_reshaped)  # (T, 21, 4)
    
    # Combine root and body quaternions: (T, 22, 4)
    quat_params = np.concatenate([root_quat[:, None, :], body_quat], axis=1)  # (T, 22, 4)
    
    # Get joint positions using forward kinematics
    if use_smpl_fk:
        # TODO: If you have SMPL model, use it here for accurate FK
        # For now, fall back to in2IN's skeleton
        use_smpl_fk = False
    
    if not use_smpl_fk:
        # Use in2IN's skeleton (approximation of SMPL's FK)
        n_raw_offsets = torch.from_numpy(HML_RAW_OFFSETS)
        skeleton = Skeleton(n_raw_offsets, HML_KINEMATIC_CHAIN, "cpu")
        skeleton._offset = skeleton._raw_offset.clone()
        joints = skeleton.forward_kinematics_np(quat_params, trans)  # (T, 22, 3)
    
    # Convert body quaternions to 6D rotations (using in2IN's function)
    body_6d = quaternion_to_cont6d_np(body_quat)  # (T, 21, 6)
    body_6d_flat = body_6d.reshape(T, -1)  # (T, 126)
    
    # Flatten joint positions
    positions_flat = joints.reshape(T, -1)  # (T, 66)
    
    # Build InterHuman format: [22*3 positions | 21*6 rotations] = (T, 192)
    motion = np.concatenate([positions_flat, body_6d_flat], axis=-1)  # (T, 192)
    
    # Process through in2IN's canonicalization pipeline
    motion_proc, root_quat_init, root_pos_init = process_motion_interhuman(
        motion, 0.001, 0, n_joints=22
    )
    
    return motion_proc, root_quat_init, root_pos_init


def salsa_pair_to_interhuman(raw_euler_poses_L, raw_trans_L, 
                             raw_euler_poses_F, raw_trans_F):
    """
    Convert a Salsa dance pair to InterHuman representation.
    
    Args:
        raw_euler_poses_L: (T, 165) - Leader SMPL poses
        raw_trans_L: (T, 3) - Leader translation
        raw_euler_poses_F: (T, 165) - Follower SMPL poses
        raw_trans_F: (T, 3) - Follower translation
    
    Returns:
        motion1_proc: (T-1, 262) - Leader processed InterHuman representation
        motion2_rel: (T-1, 262) - Follower processed InterHuman representation (relative to leader)
    """
    # Process both persons
    motion1_proc, root_quat_init1, root_pos_init1 = salsa_smpl_to_interhuman(
        raw_euler_poses_L, raw_trans_L
    )
    motion2_proc, root_quat_init2, root_pos_init2 = salsa_smpl_to_interhuman(
        raw_euler_poses_F, raw_trans_F
    )
    
    # Compute relative transform (using in2IN's functions)
    r_relative = qmul_np(root_quat_init2, qinv_np(root_quat_init1))
    angle = np.arctan2(r_relative[:, 2:3], r_relative[:, 0:1])
    xz = qrot_np(root_quat_init1, root_pos_init2 - root_pos_init1)[:, [0, 2]]
    relative = np.concatenate([angle, xz], axis=-1)[0]
    
    # Apply rigid transform to person2 (using in2IN's function)
    motion2_rel = rigid_transform(relative, motion2_proc)
    
    return motion1_proc, motion2_rel


# Example usage:
print("="*60)
print("Salsa to InterHuman Conversion Function")
print("="*60)
print("\nUsage:")
print("  motion1_proc, motion2_rel = salsa_pair_to_interhuman(")
print("      raw_euler_poses_L, raw_trans_L,")
print("      raw_euler_poses_F, raw_trans_F")
print("  )")
print("\nThis function:")
print("  1. Extracts root_orient and pose_body from raw_euler_poses")
print("  2. Converts to quaternions using in2IN's expmap_to_quaternion")
print("  3. Computes joint positions using in2IN's Skeleton FK")
print("  4. Converts to 6D rotations using in2IN's quaternion_to_cont6d_np")
print("  5. Processes through in2IN's process_motion_interhuman")
print("  6. Applies in2IN's rigid_transform for relative alignment")
print("\n✓ All functions from in2IN - ensures consistency with their pipeline!")


Salsa to InterHuman Conversion Function

Usage:
  motion1_proc, motion2_rel = salsa_pair_to_interhuman(
      raw_euler_poses_L, raw_trans_L,
      raw_euler_poses_F, raw_trans_F
  )

This function:
  1. Extracts root_orient and pose_body from raw_euler_poses
  2. Converts to quaternions using in2IN's expmap_to_quaternion
  3. Computes joint positions using in2IN's Skeleton FK
  4. Converts to 6D rotations using in2IN's quaternion_to_cont6d_np
  5. Processes through in2IN's process_motion_interhuman
  6. Applies in2IN's rigid_transform for relative alignment

✓ All functions from in2IN - ensures consistency with their pipeline!


In [64]:
# Test the function with actual Salsa data from LMDB

print("="*60)
print("Testing salsa_pair_to_interhuman with actual data")
print("="*60)

# Extract raw SMPL data from the clip we loaded earlier
T_test = min(100, clip['raw_euler_poses_L'].shape[0])
raw_euler_poses_L_test = clip['raw_euler_poses_L'][:T_test]  # (T, 165)
raw_trans_L_test = clip['raw_trans_L'][:T_test]  # (T, 3)
raw_euler_poses_F_test = clip['raw_euler_poses_F'][:T_test]  # (T, 165)
raw_trans_F_test = clip['raw_trans_F'][:T_test]  # (T, 3)

print(f"Input shapes:")
print(f"  raw_euler_poses_L: {raw_euler_poses_L_test.shape}")
print(f"  raw_trans_L: {raw_trans_L_test.shape}")
print(f"  raw_euler_poses_F: {raw_euler_poses_F_test.shape}")
print(f"  raw_trans_F: {raw_trans_F_test.shape}")

# Convert to InterHuman representation
motion1_proc_test, motion2_rel_test = salsa_pair_to_interhuman(
    raw_euler_poses_L_test, raw_trans_L_test,
    raw_euler_poses_F_test, raw_trans_F_test
)

print(f"\nOutput shapes:")
print(f"  motion1_proc: {motion1_proc_test.shape} (should be (T-1, 262))")
print(f"  motion2_rel: {motion2_rel_test.shape} (should be (T-1, 262))")

print("\n✓ Conversion successful! This function is ready to use in your dataloader.")


Testing salsa_pair_to_interhuman with actual data
Input shapes:
  raw_euler_poses_L: (100, 165)
  raw_trans_L: (100, 3)
  raw_euler_poses_F: (100, 165)
  raw_trans_F: (100, 3)

Output shapes:
  motion1_proc: (99, 262) (should be (T-1, 262))
  motion2_rel: (99, 262) (should be (T-1, 262))

✓ Conversion successful! This function is ready to use in your dataloader.


In [ ]:
# ============================================================================
# CONCLUSION: Simple function to convert Salsa data to InterHuman representation
# ============================================================================
# This function uses keypoints3d and rotmat (the approach that worked in Step 5)
# and in2IN's functions to ensure consistency with their preprocessing pipeline.

def salsa_to_interhuman(keypoints3d, rotmat, n_joints=22):
    """
    Convert Salsa keypoints3d and rotmat to InterHuman representation.
    Uses the same approach as Step 5 that produced correct results.
    
    Args:
        keypoints3d: (T, 22, 3) numpy array - Joint positions
        rotmat: (T, 498) numpy array - Rotation matrices [trans (3) | flattened_rotmats (55*9=495)]
        n_joints: int - Number of joints (default: 22 for InterHuman)
    
    Returns:
        motion_interhuman: (T-1, 262) numpy array - InterHuman processed motion
                           (after process_motion_interhuman, which reduces length by 1)
        root_quat_init: (T-1, 4) numpy array - Root quaternion at each frame
        root_pos_init: (T-1, 3) numpy array - Root position at each frame
    """
    T = keypoints3d.shape[0]
    
    # Step 1: Extract positions from keypoints3d
    positions = keypoints3d.reshape(T, -1).astype(np.float32)  # (T, 66)
    
    # Step 2: Extract rotations from rotmat (skip first 3 dims which are translation)
    rotmats_flat = rotmat[:, 3:].astype(np.float32)  # (T, 495) - skip trans
    rotmats = rotmats_flat.reshape(T, 55, 3, 3)  # (T, 55, 3, 3)
    
    # Extract first 21 body joints (joints 1-21, excluding root joint 0)
    body_rotmats = rotmats[:, 1:22, :, :]  # (T, 21, 3, 3)
    
    # Convert each 3x3 matrix to 6D representation (first 2 columns)
    rotations_6d = body_rotmats[:, :, :, :2].reshape(T, 21, 6)  # (T, 21, 6)
    rotations_6d_flat = rotations_6d.reshape(T, -1)  # (T, 126)
    
    # Step 3: Build InterHuman format: [22*3 positions | 21*6 rotations]
    motion_raw = np.concatenate([positions, rotations_6d_flat], axis=-1)  # (T, 192)
    
    # Step 4: Process through in2IN's canonicalization pipeline (using in2IN's function)
    motion_interhuman, root_quat_init, root_pos_init = process_motion_interhuman(
        motion_raw, 0.001, 0, n_joints=n_joints
    )
    
    return motion_interhuman, root_quat_init, root_pos_init


def salsa_pair_to_interhuman(keypoints3d_L, rotmat_L, keypoints3d_F, rotmat_F):
    """
    Convert a Salsa pair (Leader + Follower) to InterHuman representation with relative alignment.
    Uses keypoints3d and rotmat (the approach that worked in Step 5).
    
    Args:
        keypoints3d_L: (T, 22, 3) - Leader joint positions
        rotmat_L: (T, 498) - Leader rotation matrices
        keypoints3d_F: (T, 22, 3) - Follower joint positions
        rotmat_F: (T, 498) - Follower rotation matrices
    
    Returns:
        motion1_proc: (T-1, 262) - Leader processed motion
        motion2_rel: (T-1, 262) - Follower processed motion (relatively aligned)
    """
    # Convert both to InterHuman format using in2IN's functions
    motion1_proc, root_quat_init1, root_pos_init1 = salsa_to_interhuman(
        keypoints3d_L, rotmat_L
    )
    motion2_proc, root_quat_init2, root_pos_init2 = salsa_to_interhuman(
        keypoints3d_F, rotmat_F
    )
    
    # Compute relative transform using in2IN's functions (same as InterHuman dataset)
    r_relative = qmul_np(root_quat_init2, qinv_np(root_quat_init1))
    angle = np.arctan2(r_relative[:, 2:3], r_relative[:, 0:1])
    xz = qrot_np(root_quat_init1, root_pos_init2 - root_pos_init1)[:, [0, 2]]
    relative = np.concatenate([angle, xz], axis=-1)[0]
    
    # Apply relative transform to person 2 using in2IN's function
    motion2_rel = rigid_transform(relative, motion2_proc)
    
    return motion1_proc, motion2_rel


print("="*60)
print("CONCLUSION: Salsa to InterHuman conversion functions defined")
print("="*60)
print("Functions available:")
print("  1. salsa_to_interhuman() - Convert single person from keypoints3d/rotmat")
print("  2. salsa_pair_to_interhuman() - Convert pair with relative alignment")
print("\nThese functions use in2IN's functions:")
print("  - process_motion_interhuman() - for canonicalization")
print("  - rigid_transform(), qmul_np(), qinv_np(), qrot_np() - for relative alignment")
print("\nUses keypoints3d/rotmat approach (same as Step 5 that produced correct results).")


In [ ]:
# ============================================================================
# Test the conclusion function with Salsa keypoints3d/rotmat and visualize
# ============================================================================

print("="*60)
print("Testing salsa_pair_to_interhuman() with keypoints3d/rotmat")
print("="*60)

# Extract keypoints3d and rotmat from the clip (first 100 frames for testing)
T_test = min(100, clip['keypoints3d_L'].shape[0])
keypoints3d_L_test = clip['keypoints3d_L'][:T_test]  # (T, 22, 3)
rotmat_L_test = clip['rotmat_L'][:T_test]  # (T, 498)
keypoints3d_F_test = clip['keypoints3d_F'][:T_test]  # (T, 22, 3)
rotmat_F_test = clip['rotmat_F'][:T_test]  # (T, 498)

print(f"Input shapes:")
print(f"  keypoints3d_L: {keypoints3d_L_test.shape}")
print(f"  rotmat_L: {rotmat_L_test.shape}")
print(f"  keypoints3d_F: {keypoints3d_F_test.shape}")
print(f"  rotmat_F: {rotmat_F_test.shape}")

# Convert using the conclusion function
salsa_motion1_final, salsa_motion2_final = salsa_pair_to_interhuman(
    keypoints3d_L_test, rotmat_L_test,
    keypoints3d_F_test, rotmat_F_test
)

print(f"\nOutput shapes:")
print(f"  salsa_motion1_final: {salsa_motion1_final.shape}")
print(f"  salsa_motion2_final: {salsa_motion2_final.shape}")

# Extract joints for visualization
salsa_joints1_final = salsa_motion1_final[:, :22*3].reshape(-1, 22, 3)
salsa_joints2_final = salsa_motion2_final[:, :22*3].reshape(-1, 22, 3)

print(f"\nJoint positions for visualization:")
print(f"  salsa_joints1_final: {salsa_joints1_final.shape}")
print(f"  salsa_joints2_final: {salsa_joints2_final.shape}")

# Visualize using in2IN's plot function
from in2in.utils.plot import plot_3d_motion
from in2in.utils.paramUtil import HML_KINEMATIC_CHAIN

viz_path_final = os.path.join(viz_dir, "salsa_pair_conclusion.mp4")
print(f"\nGenerating visualization: {viz_path_final}")
plot_3d_motion(
    save_path=viz_path_final,
    kinematic_tree=HML_KINEMATIC_CHAIN,
    mp_joints=[salsa_joints1_final, salsa_joints2_final],
    title="Salsa Pair (via conclusion function)",
    figsize=(12, 12),
    fps=30,
    radius=6
)

print(f"\n✓ Visualization saved: {viz_path_final}")
print("\n" + "="*60)
print("CONCLUSION COMPLETE")
print("="*60)
print("You can now use salsa_pair_to_interhuman() in your dataloader")
print("to convert Salsa pairs to InterHuman representation consistently.")
print("This function uses keypoints3d/rotmat (the approach that worked) and")
print("in2IN's functions (process_motion_interhuman, rigid_transform) for consistency.")


In [65]:
# Step 8: Visualize both InterHuman and Salsa pairs with multiple camera angles

from in2in.utils.plot import plot_3d_motion
from in2in.utils.paramUtil import HML_KINEMATIC_CHAIN
import matplotlib
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from matplotlib.animation import FuncAnimation
from tqdm import tqdm

print("="*60)
print("Step 8: Creating visualizations with multiple camera angles")
print("="*60)

# Check that Step 7 has been run (variables should be defined in previous cell)
try:
    # Try to access the variables - if they don't exist, NameError will be raised
    _ = ih_joints1_npy.shape
    _ = ih_joints2_npy.shape
    _ = ih_joints1_smpl.shape
    _ = ih_joints2_smpl.shape
    _ = salsa_joints1.shape
    _ = salsa_joints2.shape
    print("✓ Step 7 variables found. Proceeding with visualization...")
    print(f"  InterHuman joints (from .npy): {ih_joints1_npy.shape}, {ih_joints2_npy.shape}")
    print(f"  InterHuman joints (from SMPL): {ih_joints1_smpl.shape}, {ih_joints2_smpl.shape}")
    print(f"  Salsa joints: {salsa_joints1.shape}, {salsa_joints2.shape}")
except NameError as e:
    print("="*60)
    print("ERROR: Step 7 must be run first!")
    print("="*60)
    print("The required variables are not defined. Please run Step 7 (the previous cell) first.")
    print("="*60)
    raise

# Create output directory
viz_dir = os.path.join(NOTEBOOK_DIR, "visualizations")
os.makedirs(viz_dir, exist_ok=True)

# Custom visualization function with configurable camera angle
def plot_3d_motion_custom(save_path, kinematic_tree, mp_joints, title, 
                          figsize=(12, 12), fps=30, radius=6, 
                          elev=120, azim=-90):
    """Modified version of plot_3d_motion with configurable camera angle."""
    matplotlib.use('Agg')
    
    def init():
        ax.set_xlim3d([-radius / 4, radius / 4])
        ax.set_ylim3d([0, radius / 2])
        ax.set_zlim3d([0, radius / 2])
        ax.grid(b=False)
    
    def plot_xzPlane(minx, maxx, miny, minz, maxz):
        verts = [
            [minx, miny, minz],
            [minx, miny, maxz],
            [maxx, miny, maxz],
            [maxx, miny, minz]
        ]
        xz_plane = Poly3DCollection([verts])
        xz_plane.set_facecolor((0.5, 0.5, 0.5, 0.5))
        ax.add_collection3d(xz_plane)
    
    fig = plt.figure(figsize=figsize)
    ax = fig.add_subplot(111, projection='3d')
    init()
    
    mp_offset = list(range(-len(mp_joints)//2, len(mp_joints)//2, 1))
    colors = ['red', 'blue', 'black', 'red', 'blue',
              'darkblue', 'darkblue', 'darkblue', 'darkblue', 'darkblue',
              'darkred', 'darkred', 'darkred', 'darkred', 'darkred']
    mp_colors = [[colors[i]] * 15 for i in range(len(mp_offset))]
    
    mp_data = []
    for i, joints in enumerate(mp_joints):
        data = joints.copy().reshape(len(joints), -1, 3)
        MINS = data.min(axis=0).min(axis=0)
        MAXS = data.max(axis=0).max(axis=0)
        height_offset = MINS[1]
        data[:, :, 1] -= height_offset
        trajec = data[:, 0, [0, 2]]
        mp_data.append({"joints": data, "MINS": MINS, "MAXS": MAXS, "trajec": trajec})
    
    def update(index):
        bar.update(1)
        ax.clear()
        plt.axis('off')
        ax.view_init(elev=elev, azim=azim)  # Use configurable camera angle
        ax.dist = 7.5
        ax.set_xticklabels([])
        ax.set_yticklabels([])
        ax.set_zticklabels([])
        plot_xzPlane(-3, 3, 0, -3, 3)
        
        for pid, data in enumerate(mp_data):
            for i, (chain, color) in enumerate(zip(kinematic_tree, mp_colors[pid])):
                linewidth = 3.0
                ax.plot3D(data["joints"][index, chain, 0], 
                         data["joints"][index, chain, 1], 
                         data["joints"][index, chain, 2], 
                         linewidth=linewidth, color=color, alpha=1)
    
    frame_number = min([data.shape[0] for data in mp_joints])
    bar = tqdm(total=frame_number+1, desc=f"Rendering {title}")
    ani = FuncAnimation(fig, update, frames=frame_number, interval=1000 / fps, repeat=False)
    ani.save(save_path, fps=fps, writer='ffmpeg')
    plt.close()
    bar.close()

# Camera angles: (elevation, azimuth)
# elev=120, azim=-90: top-down view
# elev=20, azim=-90: front view
# elev=20, azim=0: side view
camera_angles = [
    (120, -90, "top_down"),
    (20, -90, "front"),
    (20, 0, "side"),
]

print("\nGenerating visualizations...")

# Visualize InterHuman pair (from preprocessed .npy files - ground truth) with different angles
for elev, azim, angle_name in camera_angles:
    ih_viz_path = os.path.join(viz_dir, f"interhuman_pair_from_npy_{angle_name}.mp4")
    print(f"  InterHuman (from .npy) {angle_name} view: {ih_viz_path}")
    plot_3d_motion_custom(
        save_path=ih_viz_path,
        kinematic_tree=HML_KINEMATIC_CHAIN,
        mp_joints=[ih_joints1_npy, ih_joints2_npy],
        title=f"InterHuman Pair from .npy ({angle_name})",
        figsize=(12, 12),
        fps=30,
        radius=6,
        elev=elev,
        azim=azim
    )

# Visualize InterHuman pair (from SMPL conversion - for comparison) with different angles
for elev, azim, angle_name in camera_angles:
    ih_viz_path = os.path.join(viz_dir, f"interhuman_pair_from_smpl_{angle_name}.mp4")
    print(f"  InterHuman (from SMPL) {angle_name} view: {ih_viz_path}")
    plot_3d_motion_custom(
        save_path=ih_viz_path,
        kinematic_tree=HML_KINEMATIC_CHAIN,
        mp_joints=[ih_joints1_smpl, ih_joints2_smpl],
        title=f"InterHuman Pair from SMPL ({angle_name})",
        figsize=(12, 12),
        fps=30,
        radius=6,
        elev=elev,
        azim=azim
    )

# Visualize Salsa pair with different angles
for elev, azim, angle_name in camera_angles:
    salsa_viz_path = os.path.join(viz_dir, f"salsa_pair_{angle_name}.mp4")
    print(f"  Salsa {angle_name} view: {salsa_viz_path}")
    plot_3d_motion_custom(
        save_path=salsa_viz_path,
        kinematic_tree=HML_KINEMATIC_CHAIN,
        mp_joints=[salsa_joints1, salsa_joints2],
        title=f"Salsa Pair ({angle_name})",
        figsize=(12, 12),
        fps=30,
        radius=6,
        elev=elev,
        azim=azim
    )

# Create comparison trajectory plots
fig, axes = plt.subplots(2, 2, figsize=(16, 16))

# InterHuman trajectories (from .npy - ground truth)
ax1 = axes[0, 0]
ax1.set_aspect('equal')
ih_root1_traj_npy = ih_joints1_npy[:, 0, [0, 2]]
ih_root2_traj_npy = ih_joints2_npy[:, 0, [0, 2]]
ax1.plot(ih_root1_traj_npy[:, 0], ih_root1_traj_npy[:, 1], 'r-', linewidth=2, label='InterHuman Person 1 (from .npy)')
ax1.plot(ih_root2_traj_npy[:, 0], ih_root2_traj_npy[:, 1], 'b-', linewidth=2, label='InterHuman Person 2 (from .npy)')
ax1.plot(ih_root1_traj_npy[0, 0], ih_root1_traj_npy[0, 1], 'ro', markersize=10)
ax1.plot(ih_root2_traj_npy[0, 0], ih_root2_traj_npy[0, 1], 'bo', markersize=10)
ax1.set_xlabel('X (meters)', fontsize=12)
ax1.set_ylabel('Z (meters)', fontsize=12)
ax1.set_title('InterHuman Trajectories (from .npy - Ground Truth)', fontsize=14)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# InterHuman trajectories (from SMPL - for comparison)
ax2 = axes[0, 1]
ax2.set_aspect('equal')
ih_root1_traj_smpl = ih_joints1_smpl[:, 0, [0, 2]]
ih_root2_traj_smpl = ih_joints2_smpl[:, 0, [0, 2]]
ax2.plot(ih_root1_traj_smpl[:, 0], ih_root1_traj_smpl[:, 1], 'r-', linewidth=2, label='InterHuman Person 1 (from SMPL)')
ax2.plot(ih_root2_traj_smpl[:, 0], ih_root2_traj_smpl[:, 1], 'b-', linewidth=2, label='InterHuman Person 2 (from SMPL)')
ax2.plot(ih_root1_traj_smpl[0, 0], ih_root1_traj_smpl[0, 1], 'ro', markersize=10)
ax2.plot(ih_root2_traj_smpl[0, 0], ih_root2_traj_smpl[0, 1], 'bo', markersize=10)
ax2.set_xlabel('X (meters)', fontsize=12)
ax2.set_ylabel('Z (meters)', fontsize=12)
ax2.set_title('InterHuman Trajectories (from SMPL - Approximation)', fontsize=14)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

# Salsa trajectories
ax3 = axes[1, 0]
ax3.set_aspect('equal')
salsa_root1_traj = salsa_joints1[:, 0, [0, 2]]
salsa_root2_traj = salsa_joints2[:, 0, [0, 2]]
ax3.plot(salsa_root1_traj[:, 0], salsa_root1_traj[:, 1], 'r-', linewidth=2, label='Salsa Person 1 (Leader)')
ax3.plot(salsa_root2_traj[:, 0], salsa_root2_traj[:, 1], 'b-', linewidth=2, label='Salsa Person 2 (Follower)')
ax3.plot(salsa_root1_traj[0, 0], salsa_root1_traj[0, 1], 'ro', markersize=10)
ax3.plot(salsa_root2_traj[0, 0], salsa_root2_traj[0, 1], 'bo', markersize=10)
ax3.set_xlabel('X (meters)', fontsize=12)
ax3.set_ylabel('Z (meters)', fontsize=12)
ax3.set_title('Salsa Trajectories (Top-Down)', fontsize=14)
ax3.legend(fontsize=10)
ax3.grid(True, alpha=0.3)

# Comparison: InterHuman (.npy) vs Salsa
ax4 = axes[1, 1]
ax4.set_aspect('equal')
ax4.plot(ih_root1_traj_npy[:, 0], ih_root1_traj_npy[:, 1], 'r--', linewidth=2, alpha=0.5, label='InterHuman P1 (.npy)')
ax4.plot(ih_root2_traj_npy[:, 0], ih_root2_traj_npy[:, 1], 'b--', linewidth=2, alpha=0.5, label='InterHuman P2 (.npy)')
ax4.plot(salsa_root1_traj[:, 0], salsa_root1_traj[:, 1], 'r-', linewidth=2, label='Salsa P1 (Leader)')
ax4.plot(salsa_root2_traj[:, 0], salsa_root2_traj[:, 1], 'b-', linewidth=2, label='Salsa P2 (Follower)')
ax4.set_xlabel('X (meters)', fontsize=12)
ax4.set_ylabel('Z (meters)', fontsize=12)
ax4.set_title('Comparison: InterHuman (.npy) vs Salsa', fontsize=14)
ax4.legend(fontsize=10)
ax4.grid(True, alpha=0.3)

traj_comparison_path = os.path.join(viz_dir, "trajectory_comparison.png")
plt.savefig(traj_comparison_path, dpi=150, bbox_inches='tight')
plt.close()

print(f"\n✓ Trajectory comparison saved to: {traj_comparison_path}")

print("\n" + "="*60)
print("Visualization complete!")
print("="*60)
print(f"All visualizations saved to: {viz_dir}")
print("\nInterHuman visualizations (from preprocessed .npy - ground truth):")
for _, _, angle_name in camera_angles:
    print(f"  - interhuman_pair_from_npy_{angle_name}.mp4")
print("\nInterHuman visualizations (from SMPL conversion - for comparison):")
for _, _, angle_name in camera_angles:
    print(f"  - interhuman_pair_from_smpl_{angle_name}.mp4")
print("\nSalsa visualizations:")
for _, _, angle_name in camera_angles:
    print(f"  - salsa_pair_{angle_name}.mp4")
print(f"\n  - trajectory_comparison.png: Trajectory comparisons")


Step 8: Creating visualizations with multiple camera angles
✓ Step 7 variables found. Proceeding with visualization...
  InterHuman joints (from .npy): (98, 22, 3), (98, 22, 3)
  InterHuman joints (from SMPL): (98, 22, 3), (98, 22, 3)
  Salsa joints: (98, 22, 3), (98, 22, 3)

Generating visualizations...
  InterHuman (from .npy) top_down view: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Salsa-Agent/motion_representation/data/visualizations/interhuman_pair_from_npy_top_down.mp4


Rendering InterHuman Pair from .npy (top_down):  18%|█▊        | 18/99 [00:01<00:04, 18.13it/s]

Rendering InterHuman Pair from .npy (top_down): 100%|██████████| 99/99 [00:05<00:00, 18.68it/s]


  InterHuman (from .npy) front view: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Salsa-Agent/motion_representation/data/visualizations/interhuman_pair_from_npy_front.mp4


Rendering InterHuman Pair from .npy (front): 100%|██████████| 99/99 [00:05<00:00, 17.65it/s]


...

  Salsa front view: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Salsa-Agent/motion_representation/data/visualizations/salsa_pair_front.mp4


Rendering Salsa Pair (front): 100%|██████████| 99/99 [00:05<00:00, 17.47it/s]


  Salsa side view: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Salsa-Agent/motion_representation/data/visualizations/salsa_pair_side.mp4


Rendering Salsa Pair (side): 100%|██████████| 99/99 [00:03<00:00, 25.80it/s]



✓ Trajectory comparison saved to: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Salsa-Agent/motion_representation/data/visualizations/trajectory_comparison.png

Visualization complete!
All visualizations saved to: /local-scratch/localhome/pjomeyaz/Payam_Files/Projects/Salsa_Dance/scripts/New_2025/Salsa-Agent/motion_representation/data/visualizations

InterHuman visualizations (from preprocessed .npy - ground truth):
  - interhuman_pair_from_npy_top_down.mp4
  - interhuman_pair_from_npy_front.mp4
  - interhuman_pair_from_npy_side.mp4

InterHuman visualizations (from SMPL conversion - for comparison):
  - interhuman_pair_from_smpl_top_down.mp4
  - interhuman_pair_from_smpl_front.mp4
  - interhuman_pair_from_smpl_side.mp4

Salsa visualizations:
  - salsa_pair_top_down.mp4
  - salsa_pair_front.mp4
  - salsa_pair_side.mp4

  - trajectory_comparison.png: Trajectory comparisons
